# Identifying Gluten Grains from Protein Sequence

Wheat, barley, and rye trigger reactions in someone with celiac disease. Rice, maize, oats, quinoa, millet, and
buckwheat do not. In this project you'll test whether you can tell the two apart **from
protein sequence alone**, using only the tools from this course.


## What you'll do

The project has five parts. Each one ends with something concrete you can check.

| Part | You will... | New skill |
|---|---|---|
| **0. Orientation** | Load the data and learn what one row means | Reading a data table |
| **1. Sequence to numbers** | Write functions that turn a protein string into numbers (how much of it is proline, how much is glutamine) | Functions, `apply` |
| **2. Exploration** | Compare gluten vs non-gluten grains with a histogram and a scatter plot; find which "safe" grains look suspicious | Grouped visualization |
| **3. Is the difference real?** | Run a permutation test on the gap between the two groups | Hypothesis testing |
| **4. A sharper feature** | Count a *repeating pattern* instead of raw amounts, and watch the borderline grains separate | More string logic |
| **5. Prediction** | Build a k-nearest-neighbors classifier, test it honestly on held-out data, and see where it fails | Classification |


## Why this project

Some grains harm people with celiac disease because of a protein
called **gluten**. Gluten is hard to digest, and fragments of it set off an immune reaction
in the small intestine. Researchers study exactly how those fragments fit into an immune
receptor using 3D structure prediction. This project asks a simpler version of the same
question: *can the amino-acid makeup of a grain's proteins,
by itself, separate the harmful grains from the harmless ones?* And there's a built-in
answer key, that is, we already know which grains are which, so you can tell whether your method
works.

**This is bioinformatics.** Bioinformatics (also called computational biology) means
answering biological questions with code and statistics applied to large datasets, including
DNA, protein sequences, and clinical measurements. The core skills are the ones this course already
teaches: load and clean data, visualize it, test whether a pattern is real, and build a
model that predicts. The extra piece in this module is a little biology vocabulary, which this
notebook gives you as you go.

**As a career:** Roles that combine data skills with biology have grown alongside the size
of biological datasets, in biotechnology and pharmaceutical companies, hospital and
public-health laboratories, agricultural research, and academic labs. You do not need a
biology degree to enter the field; a data-science foundation plus curiosity is a
common path.


## The biology you need:

* A **protein** is a molecule built from a chain of small parts called **amino acids**.
  There are 20 kinds. Each is written as one capital letter:
  `A C D E F G H I K L M N P Q R S T V W Y`.
* So a protein can be written as a **string of letters**. For example, one real protein in
  our data begins:

  ```
  MAAKIFALLALHALLVSGTTAAIIPQCSLAPNAIIPQFIPPVTALGNEHLAVQAIRLQQ...
  ```

  It is a chain of 267 amino acids, one letter each, read left to right.

* Two of the letters matter here: **`P`** (the amino acid *proline*) and **`Q`**
  (*glutamine*). The storage protein in wheat, barley, and rye is unusually rich in `P`
  and `Q`, and chemically, that is why it triggers celiac disease.
* In our data, each row is one protein. The `sequence` column holds its string. The
  `gluten` column says whether that protein's grain is one of the three that trigger reactions in celiac
  disease. **Predicting `gluten` from `sequence` is the whole project.**


> **Track: `pandas`** — the library used outside the classroom. A parallel `datascience`
> version of this notebook exists; `datascience_vs_pandas.md` maps one to the other. The
> **pandas note** callouts flag places where pandas behaves differently from `datascience`.


## Setup - Run these first

The next cell loads the `check_*` helpers used after every question. They print `OK` when an
answer looks right and a short hint otherwise. They are guidance, not a grade.

They live in a separate file to keep this notebook readable. You can open it if you really
want to, but it holds the expected answers, so you will enjoy this more if you don't.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display   # shows a table mid-cell, not just as the last line
%matplotlib inline

np.random.seed(8)   # keep every run reproducible (Data 8, naturally)


In [ ]:
from checks_pandas import *


## Part 0 — Orientation


Load the data into a `DataFrame` and look at the first few rows.


In [ ]:
grains = pd.read_csv('data/grains.csv')
grains.head(8)


### What each column means

| column | what it is | do we compute from it? |
|---|---|---|
| `seq_id` | Our label for the row (`S001`, `S002`, ...) | No, it just lets us point at one specific protein. |
| `grain` | The grain this protein comes from (`wheat`, `rice`, ...) | Used for grouping, and for error analysis in Part 5. |
| `gluten` | `yes` if the grain is wheat, barley, or rye; otherwise `no` | **Yes! This is the label we try to predict.** |
| `protein_name` | The name the source database gives this protein | No calculations from it. In Part 5 it helps explain *which* proteins the model gets wrong. Ignore it until then. |
| `accession` | This is the protein's ID in **UniProt**, a public protein database | No, it's only a record of where the data came from. Paste it into uniprot.org to see the original record. |
| `sequence` | The protein itself: a string of capital letters, one per amino acid | **Yes! Every number we build comes from this column.** |

You'll see the same `protein_name` (and the same `grain`) on many rows. That's
expected - a grain has many storage-protein genes and many database entries, and
each row is one protein record.


### Question 0.1

Set `n_rows` and `n_cols` to the number of rows and columns in `grains`.


In [ ]:
n_rows = ...
n_cols = ...

check_q0_1(n_rows, n_cols)


### Question 0.2

Each row is one protein, so counting rows per grain tells you how many proteins we
have for each. Build a 2-column table, `per_grain`: one column is `grain`, the other
is the count of proteins for that grain, sorted so the largest count comes first.


In [ ]:
per_grain = ...
per_grain


> **pandas note.** `value_counts()` returns a `Series` indexed by grain, already
> sorted largest first. The `datascience` version returns a two-column `Table`
> (`grain`, `count`). Same information, different shape.


In [ ]:
check_q0_2(per_grain)


### Question 0.3

Before building any model, you need to find the **score to beat**.

**Predict first.** A friend says, "I don't need to look at any protein. I'll just
say `'no'` every time." Will your friend be right more than half the time, less than
half, or about half?

Suppose you ignored the `sequence` entirely and, for every row, just guessed whichever
`gluten` value shows up more often. How often would that be right? Decide, then compute it below.

Count the `'no'` rows and the `'yes'` rows, then set `baseline_accuracy` to the
larger count divided by the total number of rows.


In [ ]:
n_no  = ...   # number of rows where gluten == 'no'
n_yes = ...   # number of rows where gluten == 'yes'
baseline_accuracy = ...   # bigger of the two counts, divided by the total number of rows

print('gluten proteins    :', n_yes)
print('non-gluten proteins:', n_no)
print('always guess the majority label ->', baseline_accuracy)
check_q0_3(baseline_accuracy)


That number is called the **baseline** - the accuracy of a strategy that uses no real
information. It is the score any real model has to beat. You'll
compare your models to this baseline in the later parts.

**Why not just look at accuracy by itself?** Imagine a dataset where only 1 protein
in 100 is gluten. A "model" that always says `'no'` would be 99% accurate, but
completely useless, because it never finds a single gluten protein. Our data is
close to balanced, so our baseline is low, but not every dataset is like that.

**In `answer_q0_3`, either finish this sentence or write the same idea in your own
words — whichever is easier to think in:**

*"If my model only got about this % accuracy, that would mean ______, because ______."*


In [ ]:
answer_q0_3 = 'your answer here'

check_written('Q0.3 written', answer_q0_3)


## Part 1 — From a sequence to numbers

A model can't work with the raw string `'MAAKIF...IF'`. It needs **features**:
numbers that summarize a sequence. In this part you build three of them:

* `p_frac`: the fraction of the protein that is proline (`P`)
* `q_frac`: the fraction that is glutamine (`Q`)
* `pq_frac`: the two added together

Then you'll attach them to the table as new columns.

You'll start with a general helper, `count_aa(seq, aa)`. The name `aa` is short for
**amino acid**: it's just a parameter, so the one function works for any letter. In
Q1.2 you'll call it with `'P'` and `'Q'` specifically.


### Question 1.1 

Write `count_aa(seq, aa)` that returns how many times the single character `aa` occurs in the string `seq`.


In [ ]:
def count_aa(seq, aa):
    # number of times the character aa appears in the string seq
    ...

check_q1_1(count_aa)


### Question 1.2

Write `aa_fraction(seq, aa)`: the fraction of `seq` that is the character `aa` (a
number between 0 and 1). Use `count_aa` in your function.

Then write `proline_fraction(seq)` and `glutamine_fraction(seq)` by calling
`aa_fraction` with the correct letter for each amino acid.

**Hint:** string comparisons in Python are case-sensitive — `'P'` and `'p'` are
different characters, so double-check which case the `sequence` column actually uses.


In [ ]:
def aa_fraction(seq, aa):
    ...

def proline_fraction(seq):
    ...

def glutamine_fraction(seq):
    ...

check_q1_2(aa_fraction, proline_fraction, glutamine_fraction)


Here is the table again, so the column names are fresh before you add to it:


In [ ]:
grains.head(3)


### Question 1.3

Add three columns to `grains`: `p_frac`, `q_frac`, and `pq_frac`
(= `p_frac + q_frac`), computed from the `sequence` column with the functions
you just wrote.

**Hints:** a pandas `Series` (a single column) has an `apply` method that runs a
function on every value in it. Assigning to `grains['a_new_column']` adds that
column in place — no reassigning `grains` needed here, unlike the `datascience`
track.


In [ ]:
grains['p_frac'] = ...
grains['q_frac'] = ...
grains['pq_frac'] = ...

display(grains[['seq_id', 'grain', 'gluten', 'p_frac', 'q_frac', 'pq_frac']].head(5))
check_q1_3(grains)


### Question 1.4

Here's why we used a *fraction* and not a raw count. The two cells below pick out the
longest and the shortest protein in the dataset. For each, set the raw `P` count and
the proline fraction, using your functions from Q1.1 and Q1.2.


In [ ]:
longest_seq = max(grains['sequence'], key=len)
shortest_seq = min(grains['sequence'], key=len)
print('lengths:', len(longest_seq), 'vs', len(shortest_seq))


In [ ]:
long_p_count  = ...
long_p_frac   = ...
short_p_count = ...
short_p_frac  = ...

print('longest  protein:', len(longest_seq), 'residues,', long_p_count, 'are P  ->  proline fraction', round(long_p_frac, 3))
print('shortest protein:', len(shortest_seq), 'residues,', short_p_count, 'are P  ->  proline fraction', round(short_p_frac, 3))
check_q1_4(longest_seq, shortest_seq, long_p_count, long_p_frac, short_p_count, short_p_frac)


Look at the four numbers you just printed. You want to say which of these two
proteins is **more proline-rich**, and the two measures disagree. The raw count
picks one protein; the proline fraction picks the other.

So the question is not which protein wins, but which measure settles it fairly:
the raw count, or the fraction? Write one sentence in `answer_q1_4` saying which one
you would trust here, and why.


In [ ]:
answer_q1_4 = 'your answer here'

check_written('Q1.4 written', answer_q1_4)


---
**Part 1 done.** `grains` now carries `p_frac`, `q_frac`, and `pq_frac`. Part 2
uses them to compare gluten vs non-gluten grains with a histogram and a scatter
plot, and to spot which "safe" grains look suspiciously gluten-like.


## Part 2 — Exploration

Part 1 turned every protein into numbers. Before running any test or building any
model, let's *look* at those numbers. Making pictures and summaries first is called
**exploration** - you aren't proving anything yet, just seeing what the data looks like
and noticing anything odd.


### Question 2.1

A **histogram** shows how a set of numbers is spread out. It cuts the number line into
**bins** (ranges) and shows how many values fall in each one. An **overlaid** histogram
draws both groups on one chart, one set of bars per group, so you can compare them
directly.

Draw one for `pq_frac`, with one group per `gluten` value (`'yes'` and `'no'`).

First choose your bins. `np.arange(start, stop, step)` makes a list of bin edges from
`start` up to (but not including) `stop`, counting by `step`. For example,
`np.arange(0, 1, 0.25)` gives `0, 0.25, 0.5, 0.75`.

To pick the bins, first remember that `stop` has to land **past** the largest `pq_frac`, or the
proteins above it get left out of the picture. Run the cell below to see the range
the column actually covers so you know where `stop` should actually stop.

For `step`, you're choosing the width of one bin. **There is no single correct width.**
It's a choice about how to show the data, not a number you can solve for, so nothing
here will tell you the answer. **Use `step = 0.02` to start.** 


Once the plot works, come back and try `0.05` and `0.005` as your `step`. Too few bins and the two
humps smear into one, but with too many, every bin holds only one or two proteins and the shape
disappears into spikes. Seeing that trade-off is the point.


In [ ]:
# Run this first — it tells you what range your bins have to cover.
print('pq_frac runs from', round(grains['pq_frac'].min(), 3), 'to', round(grains['pq_frac'].max(), 3))


**Hint:** pandas has no one-line grouped histogram, so use a loop. The loop below
runs once per `gluten` value, with `label` set to `'no'` and then `'yes'`, and
`subset` set to the rows for that value. In the loop, call
`plt.hist(values, bins=..., alpha=0.5, density=True, label=label)`. Here,
`alpha=0.5` makes the bars see-through so you can see where the groups overlap, and
`density=True` scales each group's bars so the total area is 1, which keeps the
comparison fair when the groups have different numbers of proteins.


In [ ]:
my_bins = ...   # bin edges, made with np.arange(start, stop, step)

for label, subset in grains.groupby('gluten'):
    ...   # one histogram of subset['pq_frac'] per group, using my_bins

plt.xlabel('pq_frac')
plt.legend(title='gluten')
check_q2_1(my_bins, grains)


### Question 2.2

A **scatter plot** puts one dot on the page for each row, using one column for the
horizontal position (x) and another for the vertical position (y). Make one with
`p_frac` on the x-axis and `q_frac` on the y-axis, with the dots colored by `gluten`.
Each dot is one protein.


**Hint:** use the same kind of loop as in Q2.1, with
`plt.scatter(x_values, y_values, s=10, alpha=0.6, label=label)` inside it. Here,
`s=10` sets the dot size.


In [ ]:
for label, subset in grains.groupby('gluten'):
    ...   # one scatter call per group: p_frac on x, q_frac on y

plt.xlabel('p_frac')
plt.ylabel('q_frac')
plt.legend(title='gluten')
check_q2_2(grains)


Look at both plots together before you move on. Write one or two sentences in
`answer_q2_2`: where do the two groups overlap, and do any dots from the `no` group
sit up among the `yes` dots?


In [ ]:
answer_q2_2 = 'your answer here'

check_written('Q2.2 written', answer_q2_2)


### Question 2.3

A picture shows the whole spread. A **group mean** squeezes each group down to a single
number, which is easier to compare. Build a table `group_means` with one row for each
`gluten` value (`'no'` and `'yes'`) and the mean of `p_frac`, `q_frac`, and `pq_frac`
for each.


**Hint:** `grains.groupby('gluten')` splits the rows by `gluten` value. Pick the
columns you want in double square brackets, like `[['p_frac', 'q_frac']]`, then
call `.mean()`.


In [ ]:
group_means = ...
display(group_means)

check_q2_3(group_means)


### Question 2.4

The `no` group is really seven different grains lumped together (rice, maize, oats, and
so on). Zoom in one level: build a table `by_grain` with the mean `pq_frac` for each
`grain`, sorted so the largest mean comes first.


In [ ]:
by_grain = ...
display(by_grain)

check_q2_4(by_grain)


> **pandas note.** Selecting one column with single brackets, `['pq_frac']`, gives a
> `Series` (one labeled column), so `by_grain` is a `Series` indexed by `grain`, like
> `per_grain` in Q0.2.


### Question 2.5

Wheat, barley, and rye are three grains known to trigger celiac disease. Which
**non-gluten** grains come closest to those three gluten grains? Name them, using the
table `by_grain` you just made.


A **classifier** is a rule that predicts a label. Think about what a
classifier like "high `pq_frac` means gluten" would say about these grains, and whether it
would be right. Write one or two sentences in `answer_q2_5`.


In [ ]:
answer_q2_5 = 'your answer here'

check_written('Q2.5 written', answer_q2_5)


---
**Part 2 done.** Gluten proteins really do look different because they have a much higher share
of `P` and `Q`. But three "safe" grains sit uncomfortably close. Part 3 asks whether the
gap between the two groups is real or could have happened by chance.


## Part 3 — Is the difference real?

Part 2 found a gap: gluten grains average a much higher `pq_frac` than the rest. But a
gap in the data is not yet evidence. Split any 251 numbers into two piles at random and
the piles will differ a little, just from which numbers happened to land where.

So before reading anything into this gap, we have to rule out the dullest explanation:
that the `gluten` label has nothing to do with `pq_frac`, and a gap this size is what
random labeling produces anyway. The tool for that is a **permutation test**. We shuffle
the labels thousands of times, measure the gap each time, and see where the real gap
falls among them.


### Question 3.1

Start with the number we are trying to explain. Set `observed_diff` to the mean
`pq_frac` of the gluten proteins **minus** the mean `pq_frac` of the rest.


In [ ]:
gluten_mean = ...
safe_mean   = ...
observed_diff = ...

print('gluten mean    :', round(gluten_mean, 4))
print('non-gluten mean:', round(safe_mean, 4))
print('difference     :', round(observed_diff, 4))
check_q3_1(observed_diff, grains)


### Question 3.2

A permutation test needs a **null hypothesis**: a specific, boring story about how the
data could have come out this way with nothing interesting going on.

The dataset holds 118 gluten proteins and 133 non-gluten ones. The null hypothesis is
not about those counts; they are just how many of each we happened to collect, and they
stay fixed no matter what. It is about the
*gap* you measured in Q3.1.

So the boring story goes: the `gluten` label tells you nothing about `pq_frac`. The 251
values are what they are, and which ones ended up labeled `yes` was effectively a coin
toss — so splitting these same numbers into a group of 118 and a group of 133 would
throw up a gap like yours anyway.

Write that null hypothesis in your own words in `answer_q3_2`, in one or two sentences.


In [ ]:
answer_q3_2 = 'your answer here'

check_written('Q3.2 written', answer_q3_2)


### Question 3.3

Now act out the null hypothesis. One **trial** is: shuffle the `gluten` labels among the
proteins, leaving every `pq_frac` value exactly where it is, then recompute the
difference in means the same way you did in Q3.1. That is one difference the null
hypothesis could have produced.

Do that 5,000 times and keep every answer. (5,000 is a convention, not a derived
number. Just enough that the shape of the result settles down, few enough to finish
while you wait. This part takes about ten seconds to run.)

Write `one_simulated_difference()` so it returns a single trial's difference, then call
it 5,000 times and collect the results in `simulated_diffs`.


**Hint:** `np.random.permutation(grains['gluten'].values)` gives you the labels in a
random order as a plain array — every label exactly once, so the 118/133 split
survives it untouched.

> Use `.values` and not the `Series` itself. `grains['gluten'].sample(frac=1)` also
> shuffles, but it carries each label's original index along with it, so pandas
> quietly puts everything back where it started the moment you line it up against
> another column. A plain array has no index to re-align on.


In [ ]:
pq_values = grains['pq_frac'].values

def one_simulated_difference():
    shuffled_labels = ...
    yes_mean = ...
    no_mean  = ...
    return ...

simulated_diffs = ...   # 5000 trials, collected

print('collected', len(simulated_diffs), 'simulated differences')
check_q3_3(simulated_diffs, observed_diff)


### Question 3.4


**Step 1 — draw the null distribution.**

Your 5,000 simulated differences *are* the null distribution: the whole range of gaps
the boring story produces. Draw them as a histogram, using
`bins=np.arange(-0.10, 0.10, 0.005)`. Everything else in the cell is written for you.

Keep the plot and the check in this one cell — the check reads whatever figure the cell
just drew.


In [ ]:
...   # histogram of simulated_diffs, using the bins above

# This adds a red dot on the axis at the observed value.
plt.scatter(observed_diff, 0, color='red', s=60, zorder=3)
plt.xlabel('difference in mean pq_frac')
check_q3_4_plot(simulated_diffs)


Look at how far apart those two things are! The red dot is not out in
the tail of the null distribution — it is off the end of it entirely! Every
simulated difference is bunched near zero. Not one shuffle in 5,000 got anywhere close.

**Step 2 — put a number on it.**

The p-value is the share of simulated differences that are *at least as large as*
`observed_diff`. It answers the the question, "if the null hypothesis were true, how often would I see
a gap this big?"

Compute it in two pieces: `n_at_least`, the count, and then `p_value`, that count
divided by the number of trials. Keeping the count as its own variable is worth it —
a p-value is a *share*, always between 0 and 1, and the raw count is what makes the
paragraph after this one readable.


**Hint:** `np.count_nonzero(condition)` counts how many entries of a True/False array
are True.


In [ ]:
n_at_least = ...
p_value = ...

print('simulated differences at least as large as observed:', n_at_least, 'out of', len(simulated_diffs))
print('p-value:', p_value)
check_q3_4_p(p_value, n_at_least, simulated_diffs, observed_diff)


Your count came out as zero out of 5,000, so the p-value is exactly `0.0`. Two things
about that number are worth getting exactly right.

**First: the probability of what?** This is the easiest thing in statistics to say
backwards, so it is worth spelling out. A p-value is the probability of seeing a gap
*at least as big as yours* **in a world where the null hypothesis is true**. Your 5,000
shuffles were that world — 5,000 draws from it — and not one of them reached 0.2881.

It is **not** the probability that the null hypothesis is true. The simulation cannot
tell you that, because every trial it ran had the null built in as an assumption. You
can ask a world "how often does this happen here?" You cannot ask it "are you real?"

**Second: `0.0` does not mean impossible.** It means it did not happen in 5,000 tries.
With 5,000 trials the smallest non-zero answer available is 1/5000 = 0.0002, so the
honest way to report this is **p < 0.0002**, not **p = 0**. Running more trials would
buy you a smaller ceiling, never a zero.


### Question 3.5

A p-value answers one narrow question and gets asked to answer several others. The
paragraph above ruled out the first misreading — it is not the probability that the
null hypothesis is true. Two more are worth ruling out, and both matter for where this
project goes next.

**Does a small p-value mean the difference is big?** You have two numbers in hand:
a gap of `0.2881` and a p-value under 0.0002. They are separate facts. Which of the
two is telling you the *size* of the gap, and what is the other one telling you?

**Does it mean `pq_frac` can now identify which grain a new protein came from?** Look
back at `by_grain` from Q2.4 before you answer this one.

In `answer_q3_5`, write two or three sentences: what this result *does* let you
conclude, and what it does not.


In [ ]:
answer_q3_5 = 'your answer here'

check_written('Q3.5 written', answer_q3_5)


---
**Part 3 done.** The gap is real: random labeling does not produce it. But "real" is a
low bar — it says the difference exists, not that it is useful. Part 4 goes after a
sharper feature, one that counts not just *how much* P and Q a protein has but how they
are arranged, and watches what that does to the three borderline grains.


## Part 4 — A sharper feature

Part 3 settled that the gap is real. It did not make the gap *useful*, and Q2.4 showed
why: oat, maize, and sorghum sit between the two groups. Their storage proteins are
prolamins too, so they really are rich in P and Q, and any rule built on `pq_frac`
alone has to either miss some gluten or wrongly flag them.

So the problem is not the threshold, it's the feature. `pq_frac` counts **how much** P
and Q a protein has and ignores **how they are arranged**, and it turns out arrangement is where
gluten is actually distinctive. The immune-triggering stretches of gluten are built
from tandem repeats, long runs of `...PQPQPQ...`, while maize and sorghum have
P and Q scattered throughout the sequence instead.

In this part you'll build a feature that can tell those apart, then watch what it does to
the three borderline grains.


### Question 4.1

Write `motif_rate(seq, motif)`: the number of times `motif` appears in `seq`, per 100
residues. Dividing by length is the same move as in Q1.2 — it keeps long and short
proteins comparable — and the ×100 helps keep the numbers readable.

**Careful!** Slow down and make sure you **count overlapping copies separately.** The
string `'QQQ'` contains `'QQ'` **twice**: once starting at position 0, once starting at
position 1. They share a letter, and both count.

Python's built-in `'QQQ'.count('QQ')` returns **1**, because `str.count` skips past
each match it finds. So you cannot use it here — walk the positions yourself and check
the slice at each one.


> **Why this matters even though it looks like a technicality.** For the motif we
> actually care about, `'PQ'`, two copies can never overlap — `P` and `Q` are different
> letters — so `str.count` would give you the right answer on this dataset and you'd
> never find out. The check below tests `'QQ'` on `'QQQ'` precisely because that is the
> case that tells a correct function from a lucky one.


In [ ]:
def motif_rate(seq, motif):
    count = 0
    last_start = len(seq) - len(motif)
    for i in np.arange(last_start + 1):
        ...   # if the slice of seq starting at i matches motif, add 1 to count
    return ...   # count, per 100 residues

check_q4_1(motif_rate)


### Question 4.2

Add a `pq_motif` column to `grains`: `motif_rate(sequence, 'PQ')` for every row.

`apply` hands your function one argument — the value in the column — so it can't pass
`'PQ'` along for you. Write a small one-argument helper that calls `motif_rate` with
the motif already filled in, and apply that.


In [ ]:
def pq_motif_rate(seq):
    return ...

grains['pq_motif'] = ...

display(grains[['grain', 'gluten', 'pq_frac', 'pq_motif']].head(5))
check_q4_2(grains)


### Question 4.3

First the coarse view. Build `motif_means`: the mean `pq_frac` **and** the mean
`pq_motif` for each `gluten` value. Keeping both columns is the point — you want to see
the two features side by side, not one after the other.


In [ ]:
motif_means = ...
display(motif_means)

check_q4_3(motif_means)


Compare the two columns as *ratios*, not differences, since they are in different units.
On `pq_frac` the gluten group is a bit under 3× the rest. On `pq_motif` it is closer to
13×. The second feature separates the two groups far more sharply than the first.


### Question 4.4

Now the view that matters. Build `motif_by_grain`: mean `pq_frac` and mean `pq_motif`
for each `grain`, sorted by mean `pq_motif` with the largest first.

This is Q2.4's table with one column added, and the added column is the whole argument
of Part 4. Watch what happens to the three grains that were stuck in the middle.


In [ ]:
motif_by_grain = ...
display(motif_by_grain)

check_q4_4(motif_by_grain)


Read the two columns across for maize and sorghum. On `pq_frac` they sit at 0.27 and
0.26 against roughly 0.47 for the gluten grains and 0.12–0.14 for the clearly safe
ones — stranded in between, which is exactly the problem Q2.5 asked you to name. On
`pq_motif` they drop to 0.8, down with buckwheat and quinoa and far below wheat's 5.0.
They are not borderline on this feature at all.

Oat is the honest exception. It falls from the middle too, but only to 1.5, still a few
times higher than the safe grains. That is not a flaw in the feature — oat avenin
genuinely does carry some repeat structure, which is part of why oat's safety in celiac
disease is more contested than rice's. Hold onto that: two oat proteins are waiting in
the mystery set in Part 5.


### Question 4.5

A feature is **specific** when a high value really does mean the thing you care about,
rather than something that merely travels with it. `pq_frac` turned out not to be:
plenty of maize protein scores high on it without being gluten.

In `answer_q4_5`, write two or three sentences on why counting `PQ` repeats is more
specific to real gluten than counting P and Q separately. What can the repeat rate
see that the fraction cannot?


In [ ]:
answer_q4_5 = 'your answer here'

check_written('Q4.5 written', answer_q4_5)


---
**Part 4 done.** You now have two features that disagree about three grains, and good
reason to trust the second one. Part 5 stops arguing and tests it: build a classifier,
feed it each feature set in turn, and see which one holds up on proteins it has never
seen.
